# 📋 [K06] 군 공지 사항 요약 AI - 분대 공지 요약 카드 만들기
### 학생용 실습 워크북 & 프롬프트 설계 가이드 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K06/K06_student.ipynb)

---

## 📌 1. 프로젝트 개요 및 배경

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 일반행정·부대관리 (Level 1) |
| **학습 소요 시간** | 10개 미션 (5일 / 총 15.0시간) |
| **실습 유형** | 생성형 AI 프롬프트 설계, 가상 데이터 분석, 대조 점검 워크북 |

### 📖 프로젝트 목표
장병은 하루에도 여러 부서의 공지를 받지만, 긴 공지 속에서 기한·대상·준비물을 놓쳐 불이익을 겪는 일이 잦습니다. 이 프로젝트에서는 공개 생성형 AI로 가상 부대 공지를 요약하는 방법을 익히고, 역할 지정과 조건 제시로 프롬프트를 다듬어 '분대 공지 요약 카드'를 완성합니다. 최종적으로 누구나 다시 쓸 수 있는 나만의 공지 요약 프롬프트를 만들어, 요약 결과를 원문과 대조해 확인하는 습관까지 기릅니다.

> **⚠️ [교육용 가상 자료 원칙]**
> - 본 워크북의 모든 자료는 안전 규정에 따라 **교육용 가상 부대 자료(가상 한빛대대)**로 구성되어 있습니다.
> - 실제 군사 기밀이나 부대 세부 인적·작전 정보는 절대 외부 AI 도구에 입력하지 않습니다.

---

> **💡 [워크북 활용 안내]**
> - 본 실습은 파이썬 코딩 지식을 요구하지 않으며, 공개 생성형 AI(ChatGPT, Claude 등)와의 상호작용 및 팩트체크 검증을 기록하는 포트폴리오입니다.
> - 코드 셀의 따옴표(`""" ... """`) 사이에 프롬프트와 결과를 입력하고 실행(`Shift + Enter`)하면, 서식에 맞춰 시각적으로 자동 렌더링됩니다.
> - 📌 **본 파일은 학습자가 직접 프롬프트를 작성하고 결과를 기록하는 '학생용 실습지'입니다.**



In [ ]:
# ============================================================================
# [환경 설정 및 헬퍼 함수] 마크다운 시각화 렌더러 및 가상 공지 데이터 뷰어
# ============================================================================
from IPython.display import display, Markdown
import json
import os

# 교육용 가상 공지 데이터베이스 (N01 ~ N12, 추가 공지 A1 ~ A2)
NOTICES_DB = {
    "N01": {"date": "11-02", "dept": "대대 행정과", "title": "11월 정기 휴가 신청 접수 안내", "content": "11월 정기 휴가 신청을 11월 5일(목) 18시까지 접수합니다. 신청은 중대 행정반에 비치된 휴가 신청서를 작성하여 소속 분대장에게 제출합니다. 휴가 기간이 겹치는 인원이 중대 정원의 20%를 넘으면 선착순이 아닌 추첨으로 조정합니다. 결과는 11월 9일(월) 중대 게시판에 게시합니다. 문의: 행정과 담당관(내선 1101)."},
    "N02": {"date": "11-02", "dept": "대대 의무실", "title": "독감 예방접종 일정 안내", "content": "독감 예방접종을 11월 10일(화)부터 11월 12일(목)까지 중대별로 실시합니다. 1중대는 10일 09시, 2중대는 11일 09시, 3중대는 12일 09시에 의무실 앞으로 집합합니다. 접종 당일 발열이 있는 인원은 접종 전 의무병에게 알리고, 접종 후 30분간 의무실 대기 후 복귀합니다. 미접종자는 11월 16일(월) 추가 접종을 받습니다."},
    "N03": {"date": "11-03", "dept": "대대 군수과", "title": "동계 피복 보급 일정", "content": "동계 피복(방한 내피, 방한 장갑, 넥워머)을 11월 6일(금) 13시부터 17시까지 중대 순서대로 보급합니다. 분대장은 분대원 치수표를 11월 4일(수)까지 군수과에 제출합니다. 치수 교환은 보급 당일에만 가능하며, 이후에는 다음 달 정기 보급 때 교환합니다."},
    "N04": {"date": "11-03", "dept": "대대 정보통신실", "title": "사이버 보안 점검 및 비밀번호 변경", "content": "11월 9일(월)부터 13일(금)까지 사이버 보안 점검을 실시합니다. 모든 인원은 11월 8일(일)까지 업무용 계정 비밀번호를 변경해야 합니다. 비밀번호는 영문·숫자·특수문자를 포함해 9자리 이상으로 설정하고, 이전 비밀번호와 같으면 안 됩니다. 개인 휴대전화로 업무 자료를 촬영하거나 전송하는 행위는 금지됩니다."},
    "N05": {"date": "11-04", "dept": "대대 교육훈련과", "title": "11월 사격 훈련 일정 변경", "content": "기상 악화 예보로 11월 6일(금) 예정이던 2중대 사격 훈련을 11월 13일(금)로 연기합니다. 1중대와 3중대 일정은 변경 없습니다. 2중대는 6일에 실내 교육(사격술 예비훈련)으로 대체하며, 장소는 대대 강당입니다. 세부 시간표는 중대장 지시에 따릅니다."},
    "N06": {"date": "11-04", "dept": "대대 복지과", "title": "부대 내 복지시설 운영 시간 조정", "content": "동절기를 맞아 11월 9일(월)부터 체력단련실 운영 시간을 06시~21시로, 사이버지식정보방 운영 시간을 18시~21시 30분으로 조정합니다. 매주 수요일 14시~16시는 체력단련실 소독으로 이용할 수 없습니다. 복지매장(PX) 운영 시간은 기존과 같습니다."},
    "N07": {"date": "11-05", "dept": "대대 인사과", "title": "자기개발 지원 신청 안내", "content": "2027년 상반기 자기개발 비용 지원 신청을 11월 20일(금)까지 받습니다. 지원 대상은 자격증 응시료와 온라인 강좌 수강료이며, 1인당 연간 한도 내에서 실비를 지원합니다. 신청 시 응시 확인서 또는 수강 등록 확인서를 첨부합니다. 지원 여부는 12월 첫째 주에 개별 통보합니다."},
    "N08": {"date": "11-05", "dept": "대대 행정과", "title": "11월 정기 휴가 신청 접수 기한 연장", "content": "11월 정기 휴가 신청 접수 기한을 11월 5일(목) 18시에서 11월 6일(금) 12시로 연장합니다. 사격 훈련 일정 변경에 따른 조치입니다. 결과 게시일(11월 9일)은 변경 없습니다."},
    "N09": {"date": "11-06", "dept": "대대 안전관리관", "title": "동절기 안전사고 예방 수칙", "content": "동절기 안전사고 예방을 위해 다음 사항을 지킵니다. 생활관 내 개인 전열기 사용을 금지합니다. 차량 운행 전 타이어 상태와 결빙 구간을 확인합니다. 야외 활동 시 방한 장구를 착용하고 2인 1조로 움직입니다. 동상 의심 증상이 있으면 즉시 의무실에 알립니다."},
    "N10": {"date": "11-06", "dept": "대대 의무실", "title": "독감 예방접종 3중대 일정 변경", "content": "3중대 독감 예방접종을 11월 12일(목) 09시에서 11월 12일(목) 14시로 변경합니다. 오전 훈련 일정 조정에 따른 것입니다. 1중대와 2중대 일정은 변경 없습니다."},
    "N11": {"date": "11-09", "dept": "대대 군수과", "title": "생활관 난방기 점검", "content": "11월 11일(수)부터 13일(금)까지 생활관 난방기를 점검합니다. 점검 시간에는 생활관을 비워 두고, 귀중품은 개인 관물대에 보관합니다. 난방기 이상 소음이나 냄새가 있으면 점검 전까지 중대 행정반에 신고합니다."},
    "N12": {"date": "11-09", "dept": "대대 교육훈련과", "title": "AI 활용 교육 참가자 모집", "content": "대대 자체 AI 활용 교육 참가자를 모집합니다. 교육은 11월 23일(월)부터 4주간 매주 월요일 19시~21시에 사이버지식정보방에서 진행합니다. 중대별 5명 이내로 11월 16일(월)까지 중대 행정반에 신청합니다. 부대 자료를 외부 AI 서비스에 입력하지 않는 보안 수칙 교육이 포함됩니다."},
    "A1": {"date": "11-10", "dept": "대대 복지과", "title": "동계 체육대회 참가 신청", "content": "11월 27일(금) 대대 체육대회를 실시합니다. 종목은 축구, 족구, 줄다리기이며 중대별 참가 명단을 11월 18일(수)까지 복지과에 제출합니다. 우천 시 11월 30일(월)로 연기합니다. 문의: 복지과(내선 1503)."},
    "A2": {"date": "11-10", "dept": "대대 정보통신실", "title": "업무용 PC 보안 업데이트", "content": "11월 12일(목) 18시부터 20시까지 업무용 PC 보안 업데이트를 진행합니다. 업데이트 중에는 PC를 끄지 말고, 18시 전까지 작업 중인 문서를 저장합니다. 업데이트 후 오류가 있으면 정보통신실(내선 1401)에 신고합니다."}
}

def view_notice(notice_id):
    """가상 공지 원문을 편리하게 확인하는 함수"""
    if notice_id in NOTICES_DB:
        n = NOTICES_DB[notice_id]
        print(f"[{notice_id}] {n['title']} (작성: {n['date']} / {n['dept']})")
        print(f"본문: {n['content']}")
    else:
        print(f"존재하지 않는 공지 ID입니다: {notice_id}")

def show_result(mission_title, prompt, result):
    """학습자가 입력한 프롬프트와 AI 요약 결과를 시각적으로 렌더링하는 헬퍼 함수"""
    p_text = prompt.strip() if prompt.strip() else "(작성된 프롬프트가 없습니다)"
    r_text = result.strip() if result.strip() else "(기록된 결과가 없습니다)"
    
    md_content = f"""
---
### 📌 [{mission_title}] 실습 기록 리포트

#### 💬 입력 프롬프트
```text
{p_text}
```

#### 📋 AI 요약 결과 및 팩트체크 검증 내역
{r_text}
---
"""
    display(Markdown(md_content))

print("✅ 워크북 시각화 헬퍼 함수 및 가상 공지 데이터베이스(N01~N12, A1~A2) 준비 완료!")



---

## 🎯 미션 01. 공지 읽기의 어려움 파악하기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **공지를 읽을 때 무엇을 자주 놓치는지 파악하고, AI 요약이 도움이 될 부분을 정합니다.** 가상 공지 12건(N01~N12)을 직접 읽고 핵심 정보 5종(기한, 대상, 장소, 준비물, 연락처)을 추출하여 수작업의 취약점을 분석하고 인간-AI 분업 기준을 수립합니다.

**이번 미션 결과물:** `공지 핵심 정보 체크표, 놓친 정보 기록, AI 활용 범위 정리` 완성

### 📑 실습 서식 및 대조 점검 양식
| 공지 번호 | 기한 | 대상 | 장소 | 준비물·제출물 | 연락처 | 놓친 정보 |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| N01 | ( ) | ( ) | ( ) | ( ) | ( ) | ( ) |
| N02 | ( ) | ( ) | ( ) | ( ) | ( ) | ( ) |
| N03 | ( ) | ( ) | ( ) | ( ) | ( ) | ( ) |
| ... | ( ) | ( ) | ( ) | ( ) | ( ) | ( ) |

- **총 소요 시간**: (    )분
- **AI에게 위임할 과업 (단순 요약/추출)**: (                      )
- **사람이 직접 확인할 과업 (최종 검증/책임)**: (                      )

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정
- **평가 구분:** 필수 ([필수])
- **핵심 평가 기준:** [필수] 문제 정의·AI 적용 판단 — 공지에서 놓치기 쉬운 정보를 찾아내고 AI가 도울 부분과 사람이 확인할 부분을 구분했는가?
- **미션 PASS 조건:**
  1. 요구 산출물의 완성도 충족
  2. 원문 대조 점검을 통한 사실 검증 및 정확성 확보 (날짜·시간·숫자 무결점)
  3. 교육용 가상 데이터 활용 및 군사보안 수칙 엄수



In [ ]:
# [미션 01. 공지 읽기의 어려움 파악하기] 프롬프트 및 실습 결과 기록
mission_01_prompt = """# [TODO] 5대 행정 정보 분류 기준 및 수작업 독해 실수 패턴을 질의하는 프롬프트를 작성하세요.
# 예시: view_notice("N01") 등을 통해 공지를 확인하고 탐색 프롬프트 작성
"""

mission_01_result = """### 📋 1. 가상 공지 12건 핵심 행정 정보 체크표 작성란
| 공지 번호 | 기한 (마감일시) | 대상 | 장소 | 준비물·제출물 | 연락처 | 놓친 정보 (자가 점검) |
| :---: | :--- | :--- | :--- | :--- | :--- | :--- |
| **N01** | | | | | | |
| **N02** | | | | | | |
| **N03** | | | | | | |
| ... | | | | | | |

- **총 소요 시간**: (    )분 (1건당 평균 약 (  )분)
- **가장 놓치기 쉬웠던 정보 유형**: 

### 🤝 2. 인간-AI 역할 분업 정의서
- **AI 위임 과업 (단순 요약/추출)**: 
- **인간 직접 확인 과업 (최종 검증/책임)**: 
"""

# 렌더러 실행 (결과를 시각적으로 확인합니다)
show_result("미션 01. 공지 읽기의 어려움 파악하기", mission_01_prompt, mission_01_result)



---

## 🎯 미션 02. 요약에 쓸 AI 도구 고르기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

군 행정 공지 요약에 가장 적합한 생성형 AI 도구를 선정하기 위해 2가지 이상의 공개 AI 도구를 실증적으로 비교·평가합니다. 동일한 공지 원문(N03)을 각각 입력해 요약 결과를 추출하고, 5대 평가 기준으로 분석하여 목적과 군 보안 수칙에 맞는 도구를 선택합니다.

**이번 미션 결과물:** `도구 비교표, 선택 이유` 완성

### 📑 실습 서식 및 대조 점검 양식
| 평가 기준 | 도구 A (예: ChatGPT) | 도구 B (예: Claude / Wrtn) | 비고 / 관찰 메모 |
| :--- | :--- | :--- | :--- |
| 1. 응답 속도 | (  )점 / 5점 | (  )점 / 5점 | |
| 2. 요약 분량 (적정성) | (  )점 / 5점 | (  )점 / 5점 | |
| 3. 날짜 표기 정확성 | (  )점 / 5점 | (  )점 / 5점 | |
| 4. 한국어 자연스러움 | (  )점 / 5점 | (  )점 / 5점 | |
| 5. 보안 준수 편의성 | (  )점 / 5점 | (  )점 / 5점 | |
| **총점** | **(  )점** | **(  )점** | |

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정
- **평가 구분:** 필수 ([필수])
- **핵심 평가 기준:** [필수] AI 도구 선택 — 비교 기준을 세워 도구를 비교하고 목적과 보안 수칙에 맞는 도구를 골랐는가?
- **미션 PASS 조건:**
  1. 요구 산출물의 완성도 충족
  2. 원문 대조 점검을 통한 사실 검증 및 정확성 확보 (날짜·시간·숫자 무결점)
  3. 교육용 가상 데이터 활용 및 군사보안 수칙 엄수



In [ ]:
# [미션 02. 요약에 쓸 AI 도구 고르기] 프롬프트 및 실습 결과 기록
mission_02_prompt = """# [TODO] 동일한 공지 N03을 2개 도구에 입력하여 비교하기 위한 프롬프트를 작성하세요.
# 예시: view_notice("N03")으로 공지를 확인한 후 프롬프트 구성
"""

mission_02_result = """### 📊 도구별 5대 기준 비교 평가표 작성란
| 평가 기준 | 도구 A (      ) | 도구 B (      ) | 비고 / 관찰 메모 |
| :--- | :---: | :---: | :--- |
| 1. 응답 속도 | (  )점 / 5점 | (  )점 / 5점 | |
| 2. 요약 분량 적정성 | (  )점 / 5점 | (  )점 / 5점 | |
| 3. 날짜 표기 정확성 | (  )점 / 5점 | (  )점 / 5점 | |
| 4. 한국어 가독성 | (  )점 / 5점 | (  )점 / 5점 | |
| 5. 보안 준수 편의성 | (  )점 / 5점 | (  )점 / 5점 | |
| **총점** | **(  )점** | **(  )점** | |

### 🏆 최종 주력 도구 선정 결과 및 이유
- **선정 도구**: 
- **선정 이유 (3문장 이내)**: 
"""

# 렌더러 실행 (결과를 시각적으로 확인합니다)
show_result("미션 02. 요약에 쓸 AI 도구 고르기", mission_02_prompt, mission_02_result)



---

## 🎯 미션 03. 첫 요약 프롬프트 작성하기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

기본적인 지시문으로 공지 1건(N01)을 요약하는 첫 프롬프트를 작성하고 AI의 요약 결과를 받아봅니다. AI가 생성한 요약문에서 필수 5대 행정 정보가 어떻게 반영되었는지 확인하고, 누락되거나 왜곡된 정보를 분석합니다.

**이번 미션 결과물:** `첫 3줄 요약 프롬프트 및 누락 정보 분석표` 완성

### 📑 실습 서식 및 대조 점검 양식
- **입력 프롬프트**: "다음 공지를 3줄로 요약해 줘."
- **필수 요소 식별**: 기한 포함 여부, 대상 포함 여부, 제출처 포함 여부
- **빠진 치명적 정보**: (                                            )

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정
- **평가 구분:** 필수 ([필수])
- **핵심 평가 기준:** [필수] 첫 요약 실행 — 공지 1건을 요약하고 빠진 필수 정보(기한·대상 등)를 찾았는가?
- **미션 PASS 조건:**
  1. 요구 산출물의 완성도 충족
  2. 원문 대조 점검을 통한 사실 검증 및 정확성 확보 (날짜·시간·숫자 무결점)
  3. 교육용 가상 데이터 활용 및 군사보안 수칙 엄수



In [ ]:
# [미션 03. 첫 요약 프롬프트 작성하기] 프롬프트 및 실습 결과 기록
mission_03_prompt = """# [TODO] N01 공지를 대상으로 3줄 요약을 요청하는 기본 프롬프트를 작성하세요.
# 예시: view_notice("N01") 본문을 인용하여 기본 프롬프트 구성
"""

mission_03_result = """### 🤖 AI 기본 3줄 요약 생성 결과
(AI의 생성 답변을 여기에 붙여넣으세요)

---
### 🔍 필수 요소 식별 및 누락 정보 분석표
- **기한 포함 여부**: [ ] 포함 / [ ] 누락
- **대상 및 제출처 포함**: [ ] 포함 / [ ] 누락
- **빠진 치명적 정보 (원문 대조)**: 
"""

# 렌더러 실행 (결과를 시각적으로 확인합니다)
show_result("미션 03. 첫 요약 프롬프트 작성하기", mission_03_prompt, mission_03_result)



---

## 🎯 미션 04. 요약 결과 확인하고 바로잡기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

AI가 생성한 요약문을 원문과 문장 단위로 1:1 대조하여 오류, 왜곡, 환각(Hallucination) 현상을 식별합니다. 발견된 오류를 구체적으로 지적하는 교정 프롬프트를 작성하여 AI에게 재요청하고 무결점 요약본을 완성합니다.

**이번 미션 결과물:** `요약-원문 1:1 교차 대조표 및 교정 완료본` 완성

### 📑 실습 서식 및 대조 점검 양식
| 요약문 문장 | 원문 근거 문장 | 오류 / 왜곡 / 환각 여부 | 교정 내용 |
| :--- | :--- | :--- | :--- |
| 1행: | | [ ] 정상 / [ ] 오류 | |
| 2행: | | [ ] 정상 / [ ] 오류 | |
| 3행: | | [ ] 정상 / [ ] 오류 | |

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정
- **평가 구분:** 필수 ([필수])
- **핵심 평가 기준:** [필수] 대조 점검 — 요약과 원문을 1:1 대조해 오류나 빠진 부분을 찾아 고쳤는가?
- **미션 PASS 조건:**
  1. 요구 산출물의 완성도 충족
  2. 원문 대조 점검을 통한 사실 검증 및 정확성 확보 (날짜·시간·숫자 무결점)
  3. 교육용 가상 데이터 활용 및 군사보안 수칙 엄수



In [ ]:
# [미션 04. 요약 결과 확인하고 바로잡기] 프롬프트 및 실습 결과 기록
mission_04_prompt = """# [TODO] 미션 3에서 누락되었던 정보(추첨 조항, 결과 게시일 등)를 포함하도록 교정을 요구하는 피드백 프롬프트를 작성하세요.
"""

mission_04_result = """### 📋 1. 요약-원문 1:1 교차 대조표 작성란
| 요약문 항목 | 원문 근거 문장 | 오류 / 누락 여부 | 교정 조치 내용 |
| :--- | :--- | :---: | :--- |
| 1행: | | [ ] 정상 / [ ] 오류 | |
| 2행: | | [ ] 정상 / [ ] 오류 | |
| 3행: | | [ ] 정상 / [ ] 오류 | |

---
### ✅ 2. 교정 완료된 최종 요약 결과
(교정 프롬프트 실행 결과를 여기에 기록하세요)
"""

# 렌더러 실행 (결과를 시각적으로 확인합니다)
show_result("미션 04. 요약 결과 확인하고 바로잡기", mission_04_prompt, mission_04_result)



---

## 🎯 미션 05. 역할과 조건으로 프롬프트 개선하기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

AI에게 명확한 페르소나(분대장)를 부여하고 필수 항목 지정, 환각 금지('원문에 없는 내용은 지어내지 말 것') 등 엄격한 제약조건을 설정합니다. 개선된 프롬프트를 공지 3건(N01~N03)에 적용하여 기본 요약 대비 누락과 환각이 어떻게 줄어드는지 전후 비교표로 입증합니다.

**이번 미션 결과물:** `역할·제약조건 개선 프롬프트 및 전후 비교표` 완성

### 📑 실습 서식 및 대조 점검 양식
| 공지 번호 | 기본 3줄 요약 문제점 | 역할·조건 지정 후 개선점 | 환각 방지 확인 |
| :--- | :--- | :--- | :--- |
| N01 (휴가) | | | [ ] 확인 |
| N02 (접종) | | | [ ] 확인 |
| N03 (피복) | | | [ ] 확인 |

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정
- **평가 구분:** 필수 ([필수])
- **핵심 평가 기준:** [필수] 프롬프트 개선 — 역할·조건을 넣어 프롬프트를 개선하고 전후 차이를 비교했는가?
- **미션 PASS 조건:**
  1. 요구 산출물의 완성도 충족
  2. 원문 대조 점검을 통한 사실 검증 및 정확성 확보 (날짜·시간·숫자 무결점)
  3. 교육용 가상 데이터 활용 및 군사보안 수칙 엄수



In [ ]:
# [미션 05. 역할과 조건으로 프롬프트 개선하기] 프롬프트 및 실습 결과 기록
mission_05_prompt = """# [TODO] 역할(분대장)과 3대 제약조건(항목구분, 원문 미표기 원칙, 수치 정확 인용)을 포함한 개선 프롬프트를 작성하세요.
"""

mission_05_result = """### 📋 공지 3건(N01~N03) 개선 프롬프트 적용 결과 및 비교표
| 공지 번호 | 기본 3줄 요약 문제점 | 역할·조건 지정 후 개선점 | 환각 방지 확인 |
| :---: | :--- | :--- | :---: |
| **N01 (휴가)** | | | [ ] 확인 |
| **N02 (접종)** | | | [ ] 확인 |
| **N03 (피복)** | | | [ ] 확인 |
"""

# 렌더러 실행 (결과를 시각적으로 확인합니다)
show_result("미션 05. 역할과 조건으로 프롬프트 개선하기", mission_05_prompt, mission_05_result)



---

## 🎯 미션 06. 한눈에 보이는 요약 형식 정하기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

모바일 메신저나 상황판에 즉시 게시할 수 있는 가독성 높은 표준 카드 템플릿 슬롯(무엇을, 언제까지, 누가, 어디서, 문의)을 정의합니다. Few-shot(예시 제시) 기법을 활용하여 AI가 지정된 규격과 이모지 포맷을 엄격히 준수하도록 훈련합니다.

**이번 미션 결과물:** `모바일 표준 요약 카드 템플릿 및 Few-shot 프롬프트` 완성

### 📑 실습 서식 및 대조 점검 양식
너는 분대장이다. 아래 [예시]의 형식과 스타일을 그대로 따라서 [새 공지]를 요약 카드 형태로 작성하라.
[예시 카드] -> [새 공지 입력]

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정
- **평가 구분:** 필수 ([필수])
- **핵심 평가 기준:** [필수] 형식 통일 — 요약 형식을 정하고 Few-shot(예시)으로 형식을 통일했는가?
- **미션 PASS 조건:**
  1. 요구 산출물의 완성도 충족
  2. 원문 대조 점검을 통한 사실 검증 및 정확성 확보 (날짜·시간·숫자 무결점)
  3. 교육용 가상 데이터 활용 및 군사보안 수칙 엄수



In [ ]:
# [미션 06. 한눈에 보이는 요약 형식 정하기] 프롬프트 및 실습 결과 기록
mission_06_prompt = """# [TODO] N03을 예시(Few-shot)로 제공하고, N01, N02, N04를 동일한 카드 형태로 출력시키는 프롬프트를 작성하세요.
"""

mission_06_result = """### 📱 Few-shot 기반 모바일 요약 카드 산출물
(N01, N02, N04에 대한 카드 출력 결과를 붙여넣으세요)
"""

# 렌더러 실행 (결과를 시각적으로 확인합니다)
show_result("미션 06. 한눈에 보이는 요약 형식 정하기", mission_06_prompt, mission_06_result)



---

## 🎯 미션 07. 여러 공지를 한 번에 정리하기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

가상 공지 12건 전체를 한꺼번에 다룰 때 발생하는 컨텍스트 누락을 방지하기 위해, 공지를 청크(Chunk, 4건씩 3개 묶음)로 나누어 마감 기한을 정밀 추출합니다. 추출된 기한을 시간 순서대로 재정렬하여 분대원들이 따라야 할 주간 할 일(To-Do) 목록을 도출합니다.

**이번 미션 결과물:** `기한순(D-day) 공지 정렬표 및 주간 분대 할 일(To-Do) 목록` 완성

### 📑 실습 서식 및 대조 점검 양식
| 마감 일시 | 공지 번호 | 공지 제목 | 주요 조치 사항 |
| :--- | :--- | :--- | :--- |
| 11월 4일(수) | N03 | 동계 피복 치수표 제출 | 군수과 제출 |
| ... | | | |

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정
- **평가 구분:** 필수 ([필수])
- **핵심 평가 기준:** [필수] 다중 공지 처리 — 12건 공지에서 기한을 추출해 기한순 목록을 만들었는가?
- **미션 PASS 조건:**
  1. 요구 산출물의 완성도 충족
  2. 원문 대조 점검을 통한 사실 검증 및 정확성 확보 (날짜·시간·숫자 무결점)
  3. 교육용 가상 데이터 활용 및 군사보안 수칙 엄수



In [ ]:
# [미션 07. 여러 공지를 한 번에 정리하기] 프롬프트 및 실습 결과 기록
mission_07_prompt = """# [TODO] 12건의 공지에서 마감일시를 추출하여 시간순으로 정렬하도록 요청하는 프롬프트를 작성하세요.
"""

mission_07_result = """### 🗓️ 1. 기한순(D-day) 공지 정렬표 작성란
| 마감 일시 | 공지 번호 | 공지 제목 | 주요 조치 사항 |
| :---: | :---: | :--- | :--- |
| | | | |

---
### ✅ 2. 이번 주 분대 주간 할 일 (To-Do) 목록
- [ ] **11/04(수)**: 
- [ ] **11/05(목)**: 
- [ ] **11/06(금)**: 
- [ ] **11/08(일)**: 
"""

# 렌더러 실행 (결과를 시각적으로 확인합니다)
show_result("미션 07. 여러 공지를 한 번에 정리하기", mission_07_prompt, mission_07_result)



---

## 🎯 미션 08. 변경 공지와 빠진 정보 점검하기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

기존 공지(N01 휴가, N02 접종, N05 사격)가 나중에 발행된 변경 공지(N08 휴가연장, N10 독감시간변경)로 인해 수정된 이력을 추적합니다. 변경 사항을 누락 없이 반영하고 구버전 마감일이 잔존하지 않도록 최종 무결성을 점검합니다.

**이번 미션 결과물:** `공지 변경 이력 대조표 및 최종 무결성 점검표` 완성

### 📑 실습 서식 및 대조 점검 양식
| 변경 공지 | 원본 공지 | 기존 내용 | 변경된 최신 내용 | 변경 사유 |
| :--- | :--- | :--- | :--- | :--- |
| N08 | N01 | 11월 5일(목) 18시 | 11월 6일(금) 12시 | 사격 훈련 일정 변경 |
| N10 | N02 | 3중대 12일 09시 | 3중대 12일 14시 | 훈련 일정 조정 |

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정
- **평가 구분:** 필수 ([필수])
- **핵심 평가 기준:** [필수] 변경 반영·재점검 — 변경 공지를 찾아 내용을 고치고 빠진 정보를 보완했는가?
- **미션 PASS 조건:**
  1. 요구 산출물의 완성도 충족
  2. 원문 대조 점검을 통한 사실 검증 및 정확성 확보 (날짜·시간·숫자 무결점)
  3. 교육용 가상 데이터 활용 및 군사보안 수칙 엄수



In [ ]:
# [미션 08. 변경 공지와 빠진 정보 점검하기] 프롬프트 및 실습 결과 기록
mission_08_prompt = """# [TODO] N08(휴가연장), N10(접종변경)을 식별하고 이전 일정을 최신화하도록 지시하는 프롬프트를 작성하세요.
"""

mission_08_result = """### 📋 1. 공지 변경 이력 대조표 작성란
| 변경 공지 | 원본 공지 | 원본 내용 | 변경된 최신 내용 | 변경 사유 |
| :---: | :---: | :--- | :--- | :--- |
| N08 | N01 | | | |
| N10 | N02 | | | |

---
### 🗓️ 2. 정정 반영된 주간 To-Do 목록
(수정된 To-Do 항목을 기록하세요)
"""

# 렌더러 실행 (결과를 시각적으로 확인합니다)
show_result("미션 08. 변경 공지와 빠진 정보 점검하기", mission_08_prompt, mission_08_result)



---

## 🎯 미션 09. 나만의 공지 요약 프롬프트 만들기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

지금까지 습득한 역할(Persona), 제약조건(Constraints), Few-shot 템플릿, 변경 추적 지침을 집대성하여 재사용 가능한 '표준 공지 요약 마스터 프롬프트'를 구축합니다. 이 프롬프트를 처음 보는 신규 공지 2건(A1 체육대회, A2 PC 보안)에 수정 없이 적용하여 범용성을 검증합니다.

**이번 미션 결과물:** `표준 공지 요약 마스터 프롬프트 템플릿 및 3줄 사용 매뉴얼` 완성

### 📑 실습 서식 및 대조 점검 양식
[나만의 공지 요약 마스터 프롬프트 틀]
# 역할 / 제약조건 / 출력 서식 / 신규 공지 입력 슬롯

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정
- **평가 구분:** 필수 ([필수])
- **핵심 평가 기준:** [필수] 나만의 활용법 — 재사용 가능한 프롬프트 틀을 만들고 새 공지에 적용해 확인했는가?
- **미션 PASS 조건:**
  1. 요구 산출물의 완성도 충족
  2. 원문 대조 점검을 통한 사실 검증 및 정확성 확보 (날짜·시간·숫자 무결점)
  3. 교육용 가상 데이터 활용 및 군사보안 수칙 엄수



In [ ]:
# [미션 09. 나만의 공지 요약 프롬프트 만들기] 프롬프트 및 실습 결과 기록
mission_09_prompt = """# [TODO] 재사용 가능한 나만의 마스터 프롬프트 템플릿을 완성하고 신규 공지 A1, A2를 적용해 보세요.
"""

mission_09_result = """### 🛠️ 1. 완성된 나만의 마스터 프롬프트 템플릿
(작성한 프롬프트 틀을 여기에 기록하세요)

---
### 🧪 2. 신규 공지(A1, A2) 적용 시험 결과
(생성된 A1, A2 요약 카드를 여기에 기록하세요)

---
### 📖 3. 3줄 사용 설명서
1. 
2. 
3. 
"""

# 렌더러 실행 (결과를 시각적으로 확인합니다)
show_result("미션 09. 나만의 공지 요약 프롬프트 만들기", mission_09_prompt, mission_09_result)



---

## 🎯 미션 10. 분대 공지 요약 카드 완성하기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

프로젝트의 최종 단계로, 이번 주 공지 전체를 한눈에 볼 수 있는 '주간 분대 공지 요약 카드' 패키지를 완성합니다. 모바일 가독성, 변경 공지 정정 완료, 기한순 To-Do 목록, 그리고 AI 산출물 면책 및 공식 확인 필수 안내문(Disclaimer)을 포함하여 배포 가능한 최종본을 산출합니다.

**이번 미션 결과물:** `최종 주간 분대 공지 요약 카드 배포 패키지 (미션10, 최종 필수)` 완성

### 📑 실습 서식 및 대조 점검 양식
=================================================
📢 [한빛대대 1중대 2분대 주간 공지 브리핑 카드]
=================================================
[공지 요약 카드들]
-------------------------------------------------
🗓️ 이번 주 분대 주간 할 일 (To-Do)
-------------------------------------------------
※ 원문 확인 필수 면책 안내문
=================================================

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정
- **평가 구분:** 필수 ([필수])
- **핵심 평가 기준:** [필수] 결과물 완성 — 요약 카드가 정확하고 읽기 쉬우며 원문 확인 안내를 포함했는가?
- **미션 PASS 조건:**
  1. 요구 산출물의 완성도 충족
  2. 원문 대조 점검을 통한 사실 검증 및 정확성 확보 (날짜·시간·숫자 무결점)
  3. 교육용 가상 데이터 활용 및 군사보안 수칙 엄수



In [ ]:
# [미션 10. 분대 공지 요약 카드 완성하기] 프롬프트 및 실습 결과 기록
mission_10_prompt = """# [TODO] 최종 주간 브리핑 카드를 생성하는 프롬프트를 작성하세요.
"""

mission_10_result = """### 📢 최종 주간 분대 공지 요약 카드 작성란
(최종 완성된 요약 카드를 여기에 붙여넣으세요)

---
### 📋 수신자 1분 인지 사용성 평가표
- [ ] 신속성: 1분 이내 인지 가능
- [ ] 정확성: 연장된 일정 반영 완료
- [ ] 보안성: 가상 데이터 준수
- [ ] 신뢰성: 면책 안내문 포함
"""

# 렌더러 실행 (결과를 시각적으로 확인합니다)
show_result("미션 10. 분대 공지 요약 카드 완성하기", mission_10_prompt, mission_10_result)

